# Scoring Crediticio — Predicción de Riesgo de Impago

**Pregunta:** ¿puede predecirse, a partir de los datos de solicitud de una tarjeta de crédito (ingreso, empleo, vivienda, composición familiar), si un cliente se convertirá en mal pagador (atraso ≥60 días en algún momento de su historial)?

**Datos:** Kaggle [`rikdifos/credit-card-approval-prediction`](https://www.kaggle.com/datasets/rikdifos/credit-card-approval-prediction) — `application_record.csv` (solicitud) + `credit_record.csv` (historial de pago mensual).

Entorno: `conda activate py310`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA_DATOS = "../datos"
RUTA_RES = "../resultados"
pd.set_option("display.max_columns", 20)

## 1. Carga y EDA

In [2]:
app = pd.read_csv(f"{RUTA_DATOS}/application_record.csv")
cred = pd.read_csv(f"{RUTA_DATOS}/credit_record.csv")

print("application_record:", app.shape, "| duplicados por ID:", app["ID"].duplicated().sum())
print("credit_record:", cred.shape, "| clientes únicos:", cred["ID"].nunique())
app.isna().sum()[app.isna().sum() > 0]

application_record: (438557, 18) | duplicados por ID: 47
credit_record: (1048575, 3) | clientes únicos: 45985


OCCUPATION_TYPE    134203
dtype: int64

`OCCUPATION_TYPE` es la única columna con nulos (~30%). `credit_record` no tiene nulos.

**Etiqueta de riesgo:** mal pagador = `STATUS` en {2,3,4,5} (atraso ≥60 días) en algún mes del historial de los últimos 60 meses.

In [3]:
cred["es_malo"] = cred["STATUS"].isin(["2", "3", "4", "5"]).astype(int)
etiqueta = cred.groupby("ID")["es_malo"].max().reset_index()
print(f"clientes con historial: {etiqueta.shape[0]}, proporción mal pagador: {etiqueta['es_malo'].mean():.4f}")

clientes_cruce = set(app["ID"]) & set(etiqueta["ID"])
print(f"clientes de application_record con historial cruzado: {len(clientes_cruce)} de {app['ID'].nunique()}")

clientes con historial: 45985, proporción mal pagador: 0.0145
clientes de application_record con historial cruzado: 36457 de 438510


In [4]:
img = plt.imread(f"{RUTA_RES}/fig_01_distribucion_etiqueta.png")
plt.figure(figsize=(6, 5)); plt.imshow(img); plt.axis("off"); plt.show()

Desbalance fuerte: solo 1.45% de los clientes con historial son mal pagador. Esto condiciona todo el modelado: no se puede usar accuracy como métrica y hace falta `class_weight="balanced"`.

## 2. Limpieza y feature engineering

In [5]:
df = pd.read_csv(f"{RUTA_RES}/dataset_limpio.csv")
print("shape dataset limpio:", df.shape)
df[["EDAD_ANIOS", "ANIOS_EMPLEO", "DESEMPLEADO"]].describe().round(1)

shape dataset limpio: (36457, 20)


,EDAD_ANIOS,ANIOS_EMPLEO,DESEMPLEADO
count,36457.0,36457.0,36457.0
mean,43.7,6.0,0.2
std,11.5,6.5,0.4
min,20.5,0.0,0.0
25%,34.1,1.1,0.0
50%,42.6,4.2,0.0
75%,53.2,8.6,0.0
max,68.9,43.0,1.0


- `OCCUPATION_TYPE` nulo → categoría `Sin_declarar`.
- `DAYS_BIRTH`/`DAYS_EMPLOYED` (negativos, con centinela 365243 para desempleados) → `EDAD_ANIOS`, `ANIOS_EMPLEO`, bandera `DESEMPLEADO`.

## 3. Comparación de modelos base

In [6]:
comparacion = pd.read_csv(f"{RUTA_RES}/comparacion_modelos.csv")
comparacion

,modelo,roc_auc,f1,precision,recall
0,random_forest,0.7946,0.2439,0.1829,0.3659
1,regresion_logistica,0.5659,0.0414,0.0216,0.4878
2,arbol_decision,0.5521,0.0401,0.0209,0.5041
3,baseline_dummy,0.5000,0.0000,0.0000,0.0000


Random Forest con `class_weight="balanced"` domina claramente: ROC-AUC 0.79 vs. ~0.55-0.57 de los modelos lineales, apenas por encima del baseline dummy (0.50).

## 4. Selección por validación cruzada y evaluación final

In [7]:
modelo_final = pd.read_csv(f"{RUTA_RES}/modelo_final.csv")
modelo_final.T

,0
mejores_params,"{'clf__max_depth': None, 'clf__min_samples_lea..."
roc_auc_cv_train,0.7993
roc_auc_test,0.8144
f1_test,0.2439
precision_test,0.1829
recall_test,0.3659


In [8]:
img = plt.imread(f"{RUTA_RES}/fig_02_curva_roc.png")
plt.figure(figsize=(6, 6)); plt.imshow(img); plt.axis("off"); plt.show()

In [9]:
img = plt.imread(f"{RUTA_RES}/fig_03_matriz_confusion.png")
plt.figure(figsize=(6, 6)); plt.imshow(img); plt.axis("off"); plt.show()

Random Forest (`n_estimators=400`, `max_depth=None`, `min_samples_leaf=1`), seleccionado por `GridSearchCV` con validación cruzada 5-fold sobre ROC-AUC. El resultado en CV (0.7993) y test (0.8144) es consistente: sin señal fuerte de sobreajuste.

Con precisión de 18%, de cada 5 alertas solo 1 es un verdadero mal pagador — la matriz de confusión muestra que el modelo detecta 45 de 123 malos pagadores del test (recall 37%) a costa de falsos positivos.

## 5. Variables más importantes

In [10]:
img = plt.imread(f"{RUTA_RES}/fig_04_importancia_variables.png")
plt.figure(figsize=(8, 6)); plt.imshow(img); plt.axis("off"); plt.show()

Edad, ingreso total y antigüedad laboral concentran la mayor parte de la señal predictiva — consistente con la intuición de riesgo crediticio: clientes más jóvenes, con menor ingreso y menos antigüedad laboral tienden a tener más riesgo de impago.

## Conclusión

El modelo discrimina mal pagador de buen pagador con ROC-AUC ~0.81, pero con precisión baja (18%): útil como **filtro de revisión manual** para priorizar casos sospechosos, no para una decisión automática de rechazo de crédito.

## Limitaciones

- El conjunto de prueba tiene solo 123 positivos: las métricas de precision/recall/F1 tienen varianza alta con ese tamaño de muestra.
- La etiqueta asume que "sin atraso ≥60 días en los últimos 60 meses" equivale a "buen pagador" — ignora que un cliente con poco historial observado puede simplemente no haber tenido oportunidad de atrasarse.
- No se probaron técnicas de remuestreo (SMOTE/undersampling) ni modelos de gradient boosting; `class_weight="balanced"` fue la única estrategia de desbalance usada.